# Inspect Ingested Data Tables

This notebook helps you inspect the SQLite bronze tables used by Skimmer.

- Default DB path: `data/skimmer.db`
- Override with env var: `SKIMMER_DB_PATH`
- It lists tables, shows schema, row counts, and recent records.

In [1]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

DEFAULT_DB_PATH = Path('data/skimmer.db')
DB_PATH = Path(os.environ.get('SKIMMER_DB_PATH', DEFAULT_DB_PATH)).expanduser()

print(f'Using database: {DB_PATH.resolve()}')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')

Using database: /home/orangepi/code_projects/skimmer/data/skimmer.db


In [2]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

if tables_df.empty:
    raise RuntimeError('No tables found in the database.')

display(tables_df)

,table_name
0,bronze_socialblade_channel_stats
1,bronze_vidiq_channel_stats
2,bronze_youtube_skimmed
3,profile_queue


In [4]:
def table_schema(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"PRAGMA table_info({table_name})", connection)


def row_count(table_name: str) -> int:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    return int(pd.read_sql_query(query, connection).iloc[0]['row_count'])


def preview_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    query = f"SELECT * FROM {table_name} ORDER BY id DESC LIMIT {int(limit)}"
    df = pd.read_sql_query(query, connection)

    if 'raw_record_json' in df.columns:
        def parse_raw(value):
            if value is None:
                return None
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value

        df['raw_record_json_parsed'] = df['raw_record_json'].map(parse_raw)

    return df


In [5]:
summary_rows = []
for table_name in tables_df['table_name'].tolist():
    summary_rows.append({'table_name': table_name, 'row_count': row_count(table_name)})

summary_df = pd.DataFrame(summary_rows).sort_values(['row_count', 'table_name'], ascending=[False, True])
display(summary_df)

,table_name,row_count
2,bronze_youtube_skimmed,3739
3,profile_queue,1313
1,bronze_vidiq_channel_stats,989
0,bronze_socialblade_channel_stats,14


In [6]:
for table_name in tables_df['table_name'].tolist():
    print('\n' + '=' * 100)
    print(f'TABLE: {table_name}')
    print('=' * 100)

    print('\nSchema:')
    display(table_schema(table_name))

    print('\nLatest rows:')
    display(preview_table(table_name, limit=20))


TABLE: bronze_socialblade_channel_stats

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,subscribers_change,,0,None,0
3,3,subscribers_total,,0,None,0
4,4,views_change,,0,None,0
5,5,views_total,,0,None,0
6,6,videos_change,,0,None,0
7,7,videos_total,,0,None,0
8,8,earnings_low,,0,None,0
9,9,earnings_high,,0,None,0



Latest rows:


,id,channel_id,subscribers_change,subscribers_total,views_change,views_total,videos_change,videos_total,earnings_low,earnings_high,data_digest
0,14,@1hundred1,NaN,153000,45726,58980907,1.0,344,11,183,531a182476a98071e50f11bcd4c3203585425e226f751b...
1,13,@1hundred1,NaN,153000,98687,58935181,NaN,343,25,395,c45f9dd1ce76a05410899f80be0b1e7a6fcebae314cec7...
2,12,@1hundred1,NaN,153000,142747,58836494,NaN,343,36,571,98415a833b43b3214d2e3677a6c1c7449bbdd545ca9b81...
3,11,@1hundred1,NaN,153000,137466,58693747,NaN,343,34,550,9486069a49aa3fb88c585b6f659309cbbaf7e6255f8c3a...
4,10,@1hundred1,1000.0,153000,33812,58556281,1.0,343,8,135,b3cc0fa6877f75562586a8cac01484fbef1f609947a3c7...
5,9,@1hundred1,NaN,152000,40574,58522469,NaN,342,10,162,23dbc73d00a9fb57c49e2b28e1ebf657de6dcf51dc63b3...
6,8,@1hundred1,NaN,152000,47037,58481895,NaN,342,12,188,0aafa91c3813630ecb66f595a34ad95fc6fe75f7131790...
7,7,@1hundred1,NaN,152000,47487,58434858,1.0,342,12,190,a879725b9792779f9a6af66616f40d0d3f47f807c6db56...
8,6,@1hundred1,NaN,152000,40326,58387371,NaN,341,10,161,33e6ed492fb724a01239070e9c0ed8569a520452dfd5c5...
9,5,@1hundred1,NaN,152000,46858,58347045,NaN,341,12,187,1a07438366ff22ff351d8bd398037bce84ded433ac8a92...



TABLE: bronze_vidiq_channel_stats

Schema:


...

,id,observed_at,video_published_at,source_file,video_name,channel_display_name,views,age,channel_id,record_digest
0,3739,2026-07-20T15:58:00+00:00,2026-07-19T15:58:00+00:00,https://www.youtube.com,Preakness Champion Napoleon Solo Returns Again...,APharoah12,11K views,1 day ago,@apharoah12,c73566efc56874f3a70e420cc45248b7977dd6aebb5281...
1,3738,2026-07-20T15:58:00+00:00,2026-07-20T15:58:00+00:00,https://www.youtube.com,The World Cup Final! SPAIN vs. ARGENTINA 🏆⚽🔥 |...,Trevor Noah,606K views,Streamed 17 hours ago,@trevornoah,000d941103fab5740c884ae8561d72bb599b8e278cde7b...
2,3737,2026-07-20T15:58:00+00:00,2026-07-20T07:58:00+00:00,https://www.youtube.com,‘UNACCEPTABLE!’ 😤 Should Leandro Paredes be BA...,ESPN UK,214K views,8 hours ago,@ESPNUK,233c3b093fb6fec0f529dcf69d781bbf6d9cc3e3f837de...
3,3736,2026-07-20T15:58:00+00:00,2026-07-19T23:58:00+00:00,https://www.youtube.com,"Argentina Played Dirty and Still Lost""! - Thie...",ftblink,1M views,16 hours ago,@ftblink,bf9cc7d4981f6a66c00b77044c2976147a8bd64dc87a09...
4,3735,2026-07-20T15:58:00+00:00,2026-07-20T14:58:00+00:00,https://www.youtube.com,🚨 Trump gets polling he FEARED in key state HE...,Scott MacFarlane Reports,6.5K views,1 hour ago,@ScottMacFarlaneNews,f15df0257c1049aedea20c367b24310ab9000c4f1b5bab...
5,3734,2026-07-20T15:58:00+00:00,2026-07-19T22:58:00+00:00,https://www.youtube.com,‘It’s a really bad image for them’ 🗣️ REACTION...,ESPN FC,242K views,17 hours ago,@ESPNFC,bd26d58a5aeb6b31a6b0de28e83d70832be94c23123127...
6,3733,2026-07-20T15:58:00+00:00,2026-07-18T15:58:00+00:00,https://www.youtube.com,Jon on America's Gerontocracy Crisis & Kosta o...,The Daily Show,682K views,2 days ago,@TheDailyShow,cadd64766770eabe7bd39c602bf59db9f4428cead58a2b...
7,3732,2026-07-20T15:58:00+00:00,2026-07-18T15:58:00+00:00,https://www.youtube.com,BLUEY - Best TRY NOT TO LAUGH - COMPILATION 🤣 ...,PAW YTP,506K views,2 days ago,@PAW_YTP,7a4ab092481b47c2afc83ca231f5111a896a2c1089fa31...
8,3731,2026-07-20T15:58:00+00:00,2026-07-19T15:58:00+00:00,https://www.youtube.com,"Bluey, Bingo, and Muffin Stay Up Late! ⏰ | Ful...",Bluey - Official Channel,209K views,1 day ago,@BlueyOfficialChannel,deae08790253e7ae7d7bba62d0f63500c310cd22582b3b...
9,3730,2026-07-20T15:58:00+00:00,2026-07-12T15:58:00+00:00,https://www.youtube.com,3 Hours of One Krabby Patty Moment From EVERY ...,SpongeBob SquarePants Official,919K views,8 days ago,@SpongeBobOfficial,d2f85f9661f087c91efdaca23a3602d736524196e8bacd...



TABLE: profile_queue

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,channel_key,TEXT,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,channel_name,TEXT,0,None,0
3,3,latest_video_at,TEXT,1,None,0
4,4,last_seen_at,TEXT,1,None,0
5,5,last_success_at,TEXT,0,None,0
6,6,digested,INTEGER,1,0,0
7,7,assigned_source,TEXT,1,None,0
8,8,vidiq_failed,INTEGER,1,0,0
9,9,socialblade_failed,INTEGER,1,0,0



Latest rows:


DatabaseError: Execution failed on sql 'SELECT * FROM profile_queue ORDER BY id DESC LIMIT 20': no such column: id

In [ ]:
connection.close()